In [1]:
# Importing Modules
import pandas as pd
import itertools
from sklearn.model_selection import train_test_split
from src.dataloader import loadData
from src.model import GNNModel
from src.train_test import TrainGNN, TestGNN
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error

In [2]:
# Loading ESOL data
esol_data_train = pd.read_csv("data/processed/train/ESOL.csv")
esol_data_val = pd.read_csv("data/processed/val/ESOL.csv")

# Loading Lipophilicity data
lipophilicity_data_train = pd.read_csv("data/processed/train/Lipophilicity.csv")
lipophilicity_data_val = pd.read_csv("data/processed/val/Lipophilicity.csv")

# Loading B3DB data
b3db_data_train = pd.read_csv("data/processed/train/B3DB.csv")
b3db_data_val = pd.read_csv("data/processed/val/B3DB.csv")

# Loading RT data
rt_data_train = pd.read_csv("data/processed/train/RT.csv")
rt_data_val = pd.read_csv("data/processed/val/RT.csv")

In [3]:
# Function to run analysis
def RunGNN(data, dataName, modelName, h_dim, b_size, lr, d_out, wd, layers):
    
	train_data = data["train"]
	val_data = data["val"]
    
	# SMILES (To be used to generate molecular graph, training and testing)
	smiles_X_train = train_data["smiles"]
	X_train = smiles_X_train.to_numpy()
	smiles_X_val = val_data["smiles"]
	X_val = smiles_X_val.to_numpy()

	# Target labels
	y_train = train_data["target"]
	y_train = y_train.to_numpy()
	y_val = val_data["target"]
	y_val = y_val.to_numpy()

	# Loading dataset
	train_loader = loadData(X_train, y_train, batch_size=b_size)
	test_loader = loadData(X_val, y_val, batch_size=b_size)

	# Model
	model = GNNModel(num_features=6, hidden_dim=h_dim, model_type=modelName, dropout=d_out, num_layers=layers)

	# Model training
	model = TrainGNN(model, train_loader, test_loader, epochs=100, learning_rate=lr, w_decay=wd)

	# Model testing
	y_test, y_pred = TestGNN(model, test_loader)

	# Calculate MAE
	mae = mean_absolute_error(y_test, y_pred)

	# Calculate RMSE
	rmse = root_mean_squared_error(y_test, y_pred)

	# Return results
	return pd.DataFrame({"Data":[dataName], "Model":[modelName],
            "h_dim":[h_dim], "b_size":[b_size], "lr":[lr],
            "w_decay":[wd], "d_out":[d_out], "layers":[layers],
			"MAE":round(mae, 2), "RMSE":[round(rmse, 2)]})

In [4]:
# Function for model selection and hyperparameter tuning
def search_hyperparams(dataName):
    temp_out = []
    # Hyperparameter tuning loop
    for modelName in models:
        for i in grid_search_list:
            temp_out.append(RunGNN(data_dict[dataName], dataName, modelName, i["h_dim"], i["b_size"], i["lr"], i["dropout"], i["weight_decay"], i["layers"]))

    # Saving results
    final_df = pd.concat(temp_out, ignore_index=True)
    final_df.to_csv(f"results/Output_Hyperparameter_Optimization_GNN_{dataName}.csv", quoting=False, index=False)

    # Best parameters
    for model in models:
        temp = final_df[final_df["Model"]==model]
        print(temp.sort_values(by=["RMSE"]).head(1),"\n")

In [5]:
# Models
models = ["GCN", "GAT", "GIN", "GraphSAGE"]

# Hyperparameter dict
hyperparams = {
    'lr': [0.001, 0.0005],
    'h_dim': [32, 64],
    'b_size': [16, 32],
    'weight_decay': [1e-4, 1e-5], 
    'dropout': [0.2, 0.5],
    'layers': [2, 3, 4]
}

# Data
data_dict = {
    "ESOL": {"train": esol_data_train, "val":esol_data_val}, 
    "Lipophilicity": {"train":lipophilicity_data_train, "val":lipophilicity_data_val},
    "RT": {"train":rt_data_train, "val":rt_data_val},
    "B3DB": {"train":b3db_data_train, "val": b3db_data_val}
}

# Grid search list
keys = hyperparams.keys()
values = hyperparams.values()
grid_search_list = [dict(zip(keys, combination)) for combination in itertools.product(*values)]

In [6]:
# Params Optim for ESOL
search_hyperparams("ESOL")

    Data Model  h_dim  b_size      lr  w_decay  d_out  layers   MAE  RMSE
78  ESOL   GCN     64      16  0.0005  0.00001    0.2       2  0.85  1.08 

     Data Model  h_dim  b_size      lr  w_decay  d_out  layers   MAE  RMSE
179  ESOL   GAT     64      16  0.0005  0.00001    0.5       4  0.66  0.89 

     Data Model  h_dim  b_size      lr  w_decay  d_out  layers   MAE  RMSE
241  ESOL   GIN     32      16  0.0005   0.0001    0.2       3  0.62   0.8 

     Data      Model  h_dim  b_size     lr  w_decay  d_out  layers   MAE  RMSE
319  ESOL  GraphSAGE     64      16  0.001  0.00001    0.2       3  0.68  0.89 



In [7]:
# Params Optim for Lipophilicity
search_hyperparams("Lipophilicity")

             Data Model  h_dim  b_size     lr  w_decay  d_out  layers   MAE  \
31  Lipophilicity   GCN     64      16  0.001  0.00001    0.2       3  0.68   

    RMSE  
31  0.87   

              Data Model  h_dim  b_size     lr  w_decay  d_out  layers   MAE  \
131  Lipophilicity   GAT     64      16  0.001  0.00001    0.5       4  0.55   

     RMSE  
131  0.75   

              Data Model  h_dim  b_size      lr  w_decay  d_out  layers   MAE  \
284  Lipophilicity   GIN     64      32  0.0005  0.00001    0.2       4  0.97   

     RMSE  
284  1.16   

              Data      Model  h_dim  b_size     lr  w_decay  d_out  layers  \
299  Lipophilicity  GraphSAGE     32      16  0.001  0.00001    0.5       4   

      MAE  RMSE  
299  0.71  0.87   



In [8]:
# Params Optim for RT
search_hyperparams("RT")

  Data Model  h_dim  b_size     lr  w_decay  d_out  layers    MAE    RMSE
8   RT   GCN     32      16  0.001  0.00001    0.2       4  93.01  117.52 

    Data Model  h_dim  b_size     lr  w_decay  d_out  layers    MAE   RMSE
121   RT   GAT     64      16  0.001   0.0001    0.2       3  69.77  97.85 

    Data Model  h_dim  b_size      lr  w_decay  d_out  layers    MAE    RMSE
271   RT   GIN     64      16  0.0005  0.00001    0.2       3  77.99  108.55 

    Data      Model  h_dim  b_size     lr  w_decay  d_out  layers    MAE  \
314   RT  GraphSAGE     64      16  0.001   0.0001    0.2       4  80.01   

      RMSE  
314  108.9   



In [9]:
# Params Optim for B3DB
search_hyperparams("B3DB")

   Data Model  h_dim  b_size     lr  w_decay  d_out  layers   MAE  RMSE
2  B3DB   GCN     32      16  0.001   0.0001    0.2       4  0.53  0.64 

     Data Model  h_dim  b_size     lr  w_decay  d_out  layers  MAE  RMSE
102  B3DB   GAT     32      16  0.001  0.00001    0.2       2  0.5  0.62 

     Data Model  h_dim  b_size      lr  w_decay  d_out  layers   MAE  RMSE
242  B3DB   GIN     32      16  0.0005   0.0001    0.2       4  0.51  0.65 

     Data      Model  h_dim  b_size     lr  w_decay  d_out  layers   MAE  RMSE
289  B3DB  GraphSAGE     32      16  0.001   0.0001    0.2       3  0.46  0.61 

